# Waimea WaveCast — Các nguồn dự phòng: Open-Meteo 2005 → nay + cập nhật tăng dần + PacIOOS

**Notebook thiết kế theo `Waimea_NOAA_51201_Incremental_2005_Present(4).ipynb` bạn cung cấp**: chia CSV theo năm/tháng, lưu phản hồi thô, cache dữ liệu đã xử lý, ghi manifest/log, không xóa hay thay thế bản ghi cũ khi chạy lại.

### Phân biệt chính xác từng loại dữ liệu

| Nguồn | Phạm vi | Vai trò | Loại dữ liệu |
|---|---|---|---|
| **NOAA/NDBC 51201** | 2005–nay theo notebook chính đã chạy | **Nguồn chính** | Phao quan trắc (không chạy lại trong notebook này) |
| **Open-Meteo Historical Weather — ERA5** | 2005 → ngày gần nhất có tái phân tích (thường trễ ~5 ngày) | Dự phòng / bổ sung lịch sử | **Mô hình tái phân tích**, không phải phao |
| **Open-Meteo Marine Forecast** | Khoảng hiện tại và các giờ dự báo | Cập nhật gần nhất và lưu mọi phiên bản mô hình mới | **Mô hình dự báo**, không phải phao |
| **PacIOOS ERDDAP / CDIP station 106** | Theo dữ liệu truy xuất thực tế của ERDDAP | Kênh lấy quan trắc dự phòng | CÙNG PHAO với NOAA 51201; có thể gặp lỗi/thiếu lịch sử |

### Đặc tính lưu trữ

- Chỉ ghi file raw nếu phản hồi có **nội dung mới**; hash ngăn lưu trùng nội dung.
- Lịch sử ERA5 lưu `data-csv/YYYY/January-data-YYYY.csv`, khóa `timestamp_utc`. Chỉ append timestamp mới; không xóa file CSV cũ.
- Dự báo Marine cho phép **cùng timestamp có nhiều phiên bản khác giá trị**. `versions/*.csv` giữ `fetched_at_utc`, `valid_at_utc`, `content_hash`, và **chỉ append giá trị mới hoặc đã thay đổi**, không xóa phiên bản cũ.
- PacIOOS, nếu truy cập được, lưu raw CSV và append quan trắc theo timestamp.
- Mỗi nguồn có `run-logs/run_log.jsonl`; nguồn backfill theo tháng có thêm `manifest.json`. Năm 2005 không bị giả định là có ở mọi API: khi nguồn không cung cấp được, có log lỗi / không có dữ liệu.

**Nguồn tham khảo:**
[Open-Meteo Historical Weather](https://open-meteo.com/en/docs/historical-weather-api) · [Marine API](https://open-meteo.com/en/docs/marine-weather-api) · [OpenAPI historical](https://github.com/open-meteo/open-meteo/blob/main/openapi/historical-weather.yml) · [PacIOOS ERDDAP CDIP wave](https://pae-paha.pacioos.hawaii.edu/erddap/info/cdip_wave_agg/index.html).

> Phần dữ liệu gần thời gian thực: API công khai không cam kết có bản ghi vào đúng phút. Notebook chỉ cập nhật khi bạn chạy; sau này Scheduler trong Data Ingestion cần chạy tự động (30 phút/lần theo cấu hình đồ án).

## 1. Cài thư viện (một lần, trong đúng kernel)

In [1]:
%pip install -q requests pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


## 2. Cấu hình — mặc định tải đủ lịch sử ERA5 từ 2005

- `RUN_HISTORICAL=True`: xử lý từ 2005 đến thời điểm đã có tái phân tích (lùi 7 ngày để tránh kỳ chưa hoàn chỉnh).
- `RUN_LATEST_MARINE=True`: lấy dữ liệu cập nhật/dự báo Marine, lưu các phiên bản đã thấy.
- `PACIOOS_MODE='probe'`: kiểm thử **7 ngày gần nhất** trước; đổi `'full'` nếu thành công để tải lịch sử từng tháng, hoặc `'off'` để bỏ qua.
- Nếu muốn chạy thử nhanh trước khi backfill toàn bộ, đổi `START_YEAR=2026`; sau đó đổi lại `2005` và chạy lại — file 2026 đã có sẽ vẫn được giữ.
- Mỗi lần thực thi, notebook tạo thư mục theo `Path.cwd() / 'waimea_data' / 'other_apis'` giống hướng lưu của notebook NOAA.

In [2]:
from __future__ import annotations

from calendar import month_name, monthrange
from datetime import datetime, timezone, timedelta, date
from pathlib import Path
from hashlib import sha256
from urllib.parse import quote
from io import StringIO
import json
import math
import time
import uuid
import warnings

import requests
import pandas as pd
from IPython.display import display

LAT, LON = 21.671, -158.118
START_YEAR = 2005
RUN_HISTORICAL = True
RUN_LATEST_MARINE = True
PACIOOS_MODE = "probe"   # "off", "probe", hoặc "full" (thử probe trước)
HISTORICAL_LAG_DAYS = 7  # Dữ liệu tái phân tích trễ; đừng nhầm với real-time
MARINE_PAST_DAYS = 7
MARINE_FORECAST_DAYS = 2

REQUEST_PAUSE_SECONDS = 0.6
MAX_ATTEMPTS = 3
REQUEST_TIMEOUT = (12, 90)
MAX_CONSECUTIVE_SOURCE_ERRORS = 3  # Dừng sớm nếu endpoint cấu hình lỗi, tránh gọi hàng trăm lượt vô ích
RECHECK_FINALIZED_MONTHS = False   # True: đối soát lại nguồn cũ; dữ liệu cũ vẫn không bị xóa

# Để dùng thư mục bên cạnh notebook NOAA, đặt chung working directory trước khi chạy.
ROOT = Path.cwd() / "waimea_data" / "other_apis"
HIST_ROOT = ROOT / "open_meteo_historical"
MAR_ROOT = ROOT / "open_meteo_marine"
PAC_ROOT = ROOT / "pacioos_erddap"
for src in (HIST_ROOT, MAR_ROOT, PAC_ROOT):
    for sub in ("data-raw", "data-csv", "run-logs"):
        (src / sub).mkdir(parents=True, exist_ok=True)

NOW_UTC = pd.Timestamp.now(tz="UTC")
LAST_HIST_DATE = (NOW_UTC - pd.Timedelta(days=HISTORICAL_LAG_DAYS)).date()
HIST_START = date(START_YEAR, 1, 1)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
MARINE_URL = "https://marine-api.open-meteo.com/v1/marine"
PACIOOS_URL = "https://pae-paha.pacioos.hawaii.edu/erddap/tabledap/cdip_wave_agg.csv"

# Đây là tất cả 4 biến liên quan biển do Historical Weather OpenAPI công bố.
HIST_VARS = ["wave_height", "wave_direction", "wave_period", "sea_surface_temperature"]
# Toàn bộ biến Marine hourly trong Open-Meteo Marine API; có thể thiếu null theo khu vực và model.
MARINE_VARS = [
    "wave_height", "wave_direction", "wave_period", "wave_peak_period",
    "wind_wave_height", "wind_wave_direction", "wind_wave_period", "wind_wave_peak_period",
    "swell_wave_height", "swell_wave_direction", "swell_wave_period", "swell_wave_peak_period",
    "secondary_swell_wave_height", "secondary_swell_wave_direction", "secondary_swell_wave_period",
    "tertiary_swell_wave_height", "tertiary_swell_wave_direction", "tertiary_swell_wave_period",
    "sea_level_height_msl", "sea_surface_temperature",
    "ocean_current_velocity", "ocean_current_direction",
]
PAC_VARS = ["station_id", "time", "waveHs", "waveTp", "waveTa", "waveDp", "waveFlagPrimary"]

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "WaimeaWaveCast-educational-incremental/1.0", "Accept": "application/json,text/csv,*/*"})

print("Thư mục lưu:", ROOT)
print("Historical:", HIST_START, "→", LAST_HIST_DATE)
print("Marine:", MARINE_PAST_DAYS, "ngày qua +", MARINE_FORECAST_DAYS, "ngày dự báo")
print("PacIOOS:", PACIOOS_MODE)

Thư mục lưu: /run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data/other_apis
Historical: 2005-01-01 → 2026-10-31
Marine: 7 ngày qua + 2 ngày dự báo
PacIOOS: probe


## 3. Core — HTTP request an toàn, manifest, log và append-only CSV

In [3]:
def utc_now() -> pd.Timestamp:
    return pd.Timestamp.now(tz="UTC")


def sha_text(text: str) -> str:
    return sha256(text.encode("utf-8")).hexdigest()


def safe_json(path: Path, fallback=None):
    if not path.exists():
        return {} if fallback is None else fallback
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (ValueError, OSError) as exc:
        warnings.warn(f"Không đọc được manifest {path}: {exc}")
        return {} if fallback is None else fallback


def atomic_write(path: Path, data: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")
    try:
        tmp.write_text(data, encoding="utf-8")
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def save_manifest(root: Path, manifest: dict):
    atomic_write(root / "manifest.json", json.dumps(manifest, ensure_ascii=False, indent=2))


def log_run(root: Path, record: dict):
    # Append-only: 1 dòng cho mỗi lần gọi/chunk; không ghi đè các log trước đó.
    p = root / "run-logs" / "run_log.jsonl"
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"logged_at_utc": utc_now().isoformat(), **record}, default=str, ensure_ascii=False) + "\n")


def get_url(url: str, params=None) -> tuple[str, str]:
    """Trả (text, URL thực); không trả dữ liệu giả khi HTTP lỗi."""
    err = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            r = SESSION.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code == 404:
                raise ValueError(f"404 Not Found: {r.url}")
            if r.status_code in (429, 500, 502, 503, 504) and attempt < MAX_ATTEMPTS:
                backoff = min(30, 2 ** attempt)
                try:
                    backoff = max(backoff, min(float(r.headers.get("Retry-After", "0")), 60.0))
                except ValueError:
                    pass
                print(f"[RETRY] HTTP {r.status_code}; nghỉ {backoff:.0f}s")
                time.sleep(backoff)
                continue
            r.raise_for_status()
            if not r.content:
                raise ValueError("Phản hồi HTTP 200 nhưng body rỗng")
            return r.text, r.url
        except (requests.RequestException, ValueError) as exc:
            err = exc
            if attempt < MAX_ATTEMPTS:
                time.sleep(min(30, 2 ** attempt))
    raise RuntimeError(f"Lấy dữ liệu thất bại: {type(err).__name__}: {err}")


def raw_once(folder: Path, label: str, payload: str, extension: str = ".json") -> Path:
    """Lưu bản gốc theo hash; mỗi nội dung duy nhất được giữ lại, không ghi đè."""
    # Bỏ qua thời gian xử lý sinh ra ngẫu nhiên của Open-Meteo khi tính hash.
    # Raw gốc vẫn giữ nguyên nếu xuất hiện lần đầu.
    canonical = payload
    if extension == ".json":
        try:
            obj = json.loads(payload)
            if isinstance(obj, dict):
                obj.pop("generationtime_ms", None)
                canonical = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
        except (ValueError, TypeError):
            pass
    fingerprint = sha_text(canonical)
    name = f"{label}-{fingerprint[:16]}{extension}"
    p = folder / name
    if not p.exists():
        atomic_write(p, payload)
    return p


def month_chunks(start: date, end: date):
    """Sinh tất cả khoảng theo tháng, không giả định mọi nguồn có đủ lịch sử."""
    cursor = date(start.year, start.month, 1)
    while cursor <= end:
        month_last = date(cursor.year, cursor.month, monthrange(cursor.year, cursor.month)[1])
        begin = max(start, cursor)
        finish = min(end, month_last)
        if begin <= finish:
            yield cursor.year, cursor.month, begin, finish, month_last
        cursor = (month_last + timedelta(days=1)).replace(day=1)


def iso_frame(hourly: dict, timestamp_name: str = "time") -> pd.DataFrame:
    if not isinstance(hourly, dict) or timestamp_name not in hourly:
        raise ValueError("Không tìm thấy mảng thời gian từ API")
    count = len(hourly[timestamp_name])
    for k, values in hourly.items():
        if not isinstance(values, list) or len(values) != count:
            raise ValueError(f"Cột {k} có độ dài khác time hoặc định dạng sai")
    df = pd.DataFrame(hourly).rename(columns={timestamp_name: "timestamp_utc"})
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp_utc"]).copy()
    if df.empty:
        return df
    df["timestamp_utc"] = df["timestamp_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return df.drop_duplicates(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)


def append_csv_unique(frame: pd.DataFrame, path: Path, key: str = "timestamp_utc") -> tuple[int, int]:
    """Chỉ nối bản ghi mới. Không ghi lại hoặc xóa file cũ; khóa unique được kiểm tra.

    Nếu các cột API thay đổi, dừng và yêu cầu migration thay vì làm lệch header CSV.
    """
    if frame.empty:
        return 0, 0
    frame = frame.copy()
    if key not in frame:
        raise ValueError(f"Thiếu unique key: {key}")
    frame = frame.drop_duplicates(subset=[key]).sort_values(key)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size:
        old_header = pd.read_csv(path, nrows=0).columns.tolist()
        if set(old_header) != set(frame.columns):
            raise ValueError(f"Schema drift tại {path.name}; cần migration: old={old_header}, new={frame.columns.tolist()}")
        frame = frame.reindex(columns=old_header)
        seen = set(pd.read_csv(path, usecols=[key], dtype=str)[key].dropna().astype(str))
        fresh = frame.loc[~frame[key].astype(str).isin(seen)].copy()
        if len(fresh):
            fresh.to_csv(path, mode="a", index=False, header=False, encoding="utf-8")
    else:
        fresh = frame
        fresh.to_csv(path, mode="x" if not path.exists() else "w", index=False, encoding="utf-8")
    return len(fresh), len(frame)


def month_path(base: Path, year: int, month: int, label: str = "data") -> Path:
    return base / "data-csv" / str(year) / f"{month_name[month]}-{label}-{year}.csv"

## 4. Tự kiểm thử append-only trước khi gọi API thật

Tạo dữ liệu **giả lập chỉ trong thư mục tạm của Python** để kiểm tra không ghi trùng và phát hiện schema drift. Dữ liệu giả này không được lưu vào `waimea_data` và không tham gia thống kê dự án.

In [4]:
from tempfile import TemporaryDirectory
with TemporaryDirectory() as td:
    test_file = Path(td) / "test.csv"
    a = pd.DataFrame({"timestamp_utc": ["2005-01-01T00:00:00Z", "2005-01-01T01:00:00Z"], "wave_height": [1.2, 1.4]})
    assert append_csv_unique(a, test_file) == (2, 2)
    assert append_csv_unique(a, test_file) == (0, 2)
    b = pd.DataFrame({"timestamp_utc": ["2005-01-01T01:00:00Z", "2005-01-01T02:00:00Z"], "wave_height": [999.0, 1.5]})
    assert append_csv_unique(b, test_file) == (1, 2)
    final = pd.read_csv(test_file)
    assert len(final) == 3 and final.iloc[1]["wave_height"] == 1.4
print("PASS: chạy lại không tạo trùng, dữ liệu cũ không bị thay bằng bản mới.")

PASS: chạy lại không tạo trùng, dữ liệu cũ không bị thay bằng bản mới.


## 5. Open-Meteo Historical Weather (ERA5) — tải 2005 → hiện tại, CSV từng tháng

API `archive-api.open-meteo.com/v1/archive`: chỉ yêu cầu **4 biến biển có trong tài liệu Historical Weather**; không tự gán 22 biến từ Marine Forecast cho năm 2005. API có thể có dữ liệu thiếu ở một địa điểm. `models=era5` nhằm ưu tiên tính nhất quán cho lịch sử dài hạn; nếu dịch vụ báo không hỗ trợ biến, log lỗi và dừng sớm để người dùng kiểm tra, không âm thầm đổi nguồn.

Dữ liệu đến `NOW_UTC - 7 ngày`: các ngày mới nhất thuộc `Open-Meteo Marine` bên dưới. Với mỗi tháng hoàn tất, đánh dấu `complete=true` trong manifest để lần chạy sau không tải lại. Tháng còn mở sẽ tiếp tục đọc và append các timestamp mới.

In [5]:
def parse_historical_payload(payload: dict) -> pd.DataFrame:
    if payload.get("error"):
        raise ValueError(str(payload.get("reason", "API trả error=true")))
    raw_hourly = payload.get("hourly")
    frame = iso_frame(raw_hourly)
    if frame.empty:
        raise ValueError("Không có hàng hourly hợp lệ")
    missing = set(HIST_VARS) - set(frame.columns)
    if missing:
        raise ValueError(f"API không cung cấp các cột lịch sử mong đợi: {sorted(missing)}")
    frame["source"] = "open_meteo_era5"
    frame["data_type"] = "reanalysis_model"
    frame["latitude_requested"] = LAT
    frame["longitude_requested"] = LON
    return frame


def get_historical_month(year: int, month: int, begin: date, finish: date) -> dict:
    key = f"{year:04d}-{month:02d}"
    manifest = safe_json(HIST_ROOT / "manifest.json", {"months": {}})
    manifest.setdefault("months", {})
    entry = manifest["months"].get(key, {})
    if entry.get("complete") and not RECHECK_FINALIZED_MONTHS:
        return {"period": key, "status": "cached", "new_rows": 0}
    params = {
        "latitude": LAT, "longitude": LON,
        "start_date": begin.isoformat(), "end_date": finish.isoformat(),
        "hourly": ",".join(HIST_VARS),
        "models": "era5", "timezone": "UTC", "cell_selection": "sea",
    }
    try:
        text, url = get_url(ARCHIVE_URL, params)
        payload = json.loads(text)
        frame = parse_historical_payload(payload)
        raw = raw_once(HIST_ROOT / "data-raw" / str(year), key, text)
        frame["source_dataset"] = "historical_weather_era5"
        out = month_path(HIST_ROOT, year, month)
        new_count, incoming = append_csv_unique(frame, out)
        # Chỉ finalize tháng nếu đã yêu cầu trọn tháng VÀ API có dữ liệu đến ngày kết thúc.
        last_point = pd.to_datetime(frame["timestamp_utc"], utc=True).max()
        expected_hours = (finish - begin).days * 24 + 24
        expected_last = pd.Timestamp(finish, tz="UTC") + pd.Timedelta(hours=23)
        complete = (finish == date(year, month, monthrange(year, month)[1]) and
                    last_point >= expected_last and
                    frame["timestamp_utc"].nunique() >= expected_hours)
        manifest["months"][key] = {
            "complete": bool(complete),
            "last_success_utc": utc_now().isoformat(),
            "latest_timestamp_utc": frame["timestamp_utc"].max(),
            "rows_in_response": incoming,
            "csv_path": str(out.relative_to(HIST_ROOT)),
        }
        save_manifest(HIST_ROOT, manifest)
        record = {"period": key, "status": "ok", "new_rows": new_count, "rows_response": incoming,
                  "complete": complete, "raw": str(raw.relative_to(HIST_ROOT)), "url": url}
        log_run(HIST_ROOT, record)
        return record
    except (requests.RequestException, RuntimeError, ValueError, KeyError, OSError) as exc:
        record = {"period": key, "status": "error", "new_rows": 0,
                  "error": f"{type(exc).__name__}: {str(exc)[:320]}"}
        log_run(HIST_ROOT, record)
        return record


def run_historical():
    stats = []
    errors = 0
    if HIST_START > LAST_HIST_DATE:
        print("Không có khoảng dữ liệu lịch sử đã đủ độ trễ để tải")
        return pd.DataFrame()
    for year, month, start, end, month_end in month_chunks(HIST_START, LAST_HIST_DATE):
        result = get_historical_month(year, month, start, end)
        stats.append(result)
        if result["status"] == "error":
            errors += 1
            print("[HISTORY ERROR]", result["period"], result.get("error"))
            if errors >= MAX_CONSECUTIVE_SOURCE_ERRORS:
                print("[STOP] Dừng lịch sử vì nhiều lỗi liên tiếp; kiểm tra endpoint/biến/giới hạn API trước.")
                break
        else:
            errors = 0
            print(f"[HISTORY {result['status'].upper()}] {result['period']}: +{result['new_rows']} bản ghi")
        if result["status"] == "ok" and REQUEST_PAUSE_SECONDS:
            time.sleep(REQUEST_PAUSE_SECONDS)
    return pd.DataFrame(stats)

In [6]:
if RUN_HISTORICAL:
    historical_results = run_historical()
    if not historical_results.empty:
        display(historical_results.tail(20))
        print("Tổng hàng lịch sử mới:", int(historical_results["new_rows"].sum()))
else:
    print("RUN_HISTORICAL=False: bỏ qua backfill")

[HISTORY CACHED] 2005-01: +0 bản ghi
[HISTORY CACHED] 2005-02: +0 bản ghi
[HISTORY CACHED] 2005-03: +0 bản ghi
[HISTORY CACHED] 2005-04: +0 bản ghi
[HISTORY CACHED] 2005-05: +0 bản ghi
[HISTORY CACHED] 2005-06: +0 bản ghi
[HISTORY CACHED] 2005-07: +0 bản ghi
[HISTORY CACHED] 2005-08: +0 bản ghi
[HISTORY CACHED] 2005-09: +0 bản ghi
[HISTORY CACHED] 2005-10: +0 bản ghi
[HISTORY CACHED] 2005-11: +0 bản ghi
[HISTORY CACHED] 2005-12: +0 bản ghi
[HISTORY CACHED] 2006-01: +0 bản ghi
[HISTORY CACHED] 2006-02: +0 bản ghi
[HISTORY CACHED] 2006-03: +0 bản ghi
[HISTORY CACHED] 2006-04: +0 bản ghi
[HISTORY CACHED] 2006-05: +0 bản ghi
[HISTORY CACHED] 2006-06: +0 bản ghi
[HISTORY CACHED] 2006-07: +0 bản ghi
[HISTORY CACHED] 2006-08: +0 bản ghi
[HISTORY CACHED] 2006-09: +0 bản ghi
[HISTORY CACHED] 2006-10: +0 bản ghi
[HISTORY CACHED] 2006-11: +0 bản ghi
[HISTORY CACHED] 2006-12: +0 bản ghi
[HISTORY CACHED] 2007-01: +0 bản ghi
[HISTORY CACHED] 2007-02: +0 bản ghi
[HISTORY CACHED] 2007-03: +0 bản ghi
[

,period,status,new_rows
242,2025-03,cached,0
243,2025-04,cached,0
244,2025-05,cached,0
245,2025-06,cached,0
246,2025-07,cached,0
247,2025-08,cached,0
248,2025-09,cached,0
249,2025-10,cached,0
250,2025-11,cached,0
251,2025-12,cached,0


Tổng hàng lịch sử mới: 0


## 6. Open-Meteo Marine — cập nhật gần nhất, **giữ các phiên bản dự báo**

API Marine có nhiều biến hơn archive. Không gộp **quan trắc** với **mô hình dự báo**. Cùng `valid_at_utc` có thể thay đổi giá trị sau một lần cập nhật mô hình: notebook lưu bản mới ở `versions/YYYY/Month-versions-YYYY.csv` kèm `fetched_at_utc` và `content_hash`, giữ bản cũ để so sánh độ ổn định dự báo.

Lần chạy thứ hai với **nội dung y hệt** không tạo thêm phiên bản. Nếu API có bản dự báo thay đổi, append bản mới. Muốn lấy gần thời gian thực 30 phút/lần, sau này chuyển hàm `update_latest_marine()` sang service Scheduler của Backend.

In [7]:
def marine_fingerprint(row: dict, fields: list[str]) -> str:
    """Hash nội dung vật lý; loại bỏ fetched_at vì không phải một thay đổi dự báo."""
    obj = {k: (None if pd.isna(row.get(k)) else row.get(k)) for k in fields}
    return sha_text(json.dumps(obj, sort_keys=True, default=str, separators=(",", ":")))


def update_latest_marine() -> dict:
    params = {
        "latitude": LAT, "longitude": LON,
        "hourly": ",".join(MARINE_VARS),
        "past_days": MARINE_PAST_DAYS,
        "forecast_days": MARINE_FORECAST_DAYS,
        "timezone": "UTC", "cell_selection": "sea",
    }
    try:
        received, url = get_url(MARINE_URL, params)
        payload = json.loads(received)
        if payload.get("error"):
            raise ValueError(payload.get("reason", "Open-Meteo Marine API error"))
        frame = iso_frame(payload.get("hourly"))
        if frame.empty:
            raise ValueError("Marine API không có hourly data")
        missing = sorted(set(MARINE_VARS) - set(frame.columns))
        if missing:
            raise ValueError(f"Marine API thiếu biến đã yêu cầu: {missing}")
        fetched_at = utc_now().isoformat()
        raw = raw_once(MAR_ROOT / "data-raw" / str(utc_now().year) / f"{utc_now().month:02d}", "marine", received)

        frame["valid_at_utc"] = frame.pop("timestamp_utc")
        frame["fetched_at_utc"] = fetched_at
        frame["source"] = "open_meteo_marine"
        frame["data_type"] = "forecast_model"
        frame["forecast_status"] = frame["valid_at_utc"].map(
            lambda text: "future" if pd.Timestamp(text) > utc_now() else "current_or_past"
        )
        field_names = [c for c in frame.columns if c in MARINE_VARS]
        frame["content_hash"] = [marine_fingerprint(row, field_names) for row in frame.to_dict("records")]
        frame["revision_key"] = frame["valid_at_utc"] + "|" + frame["content_hash"]
        frame["raw_uri_local"] = str(raw.relative_to(ROOT))
        frame["valid_at"] = pd.to_datetime(frame["valid_at_utc"], utc=True)
        added = 0
        for (yr, mo), chunk in frame.groupby([frame["valid_at"].dt.year, frame["valid_at"].dt.month]):
            chunk = chunk.drop(columns=["valid_at"])
            target = month_path(MAR_ROOT, int(yr), int(mo), "versions")
            plus, _ = append_csv_unique(chunk, target, key="revision_key")
            added += plus
        result = {
            "status": "ok", "new_versions": added,
            "rows_in_response": len(frame), "raw": str(raw.relative_to(ROOT)),
            "latest_valid_at_utc": frame["valid_at_utc"].max(), "url": url,
        }
        log_run(MAR_ROOT, result)
        return result
    except (requests.RequestException, RuntimeError, ValueError, KeyError, OSError) as exc:
        result = {"status": "error", "new_versions": 0, "error": f"{type(exc).__name__}: {str(exc)[:320]}"}
        log_run(MAR_ROOT, result)
        return result


def latest_marine_view(year: int, month: int) -> pd.DataFrame:
    """Không sửa file versions: tạo view mới nhất theo valid_at khi đọc."""
    p = month_path(MAR_ROOT, year, month, "versions")
    if not p.exists():
        return pd.DataFrame()
    d = pd.read_csv(p, dtype={"valid_at_utc": str, "fetched_at_utc": str})
    return d.sort_values("fetched_at_utc").drop_duplicates("valid_at_utc", keep="last").sort_values("valid_at_utc")

In [8]:
if RUN_LATEST_MARINE:
    marine_result = update_latest_marine()
    print("Marine result:", marine_result)
    if marine_result["status"] == "ok":
        current_marine = latest_marine_view(utc_now().year, utc_now().month)
        if not current_marine.empty:
            display(current_marine[["valid_at_utc", "fetched_at_utc", "wave_height", "wave_period", "wave_direction", "forecast_status"]].tail(12))
else:
    print("RUN_LATEST_MARINE=False")

Marine result: {'status': 'ok', 'new_versions': 216, 'rows_in_response': 216, 'raw': 'open_meteo_marine/data-raw/2026/11/marine-48fbdab53fdfdc54.json', 'latest_valid_at_utc': '2026-11-08T23:00:00Z', 'url': 'https://marine-api.open-meteo.com/v1/marine?latitude=21.671&longitude=-158.118&hourly=wave_height%2Cwave_direction%2Cwave_period%2Cwave_peak_period%2Cwind_wave_height%2Cwind_wave_direction%2Cwind_wave_period%2Cwind_wave_peak_period%2Cswell_wave_height%2Cswell_wave_direction%2Cswell_wave_period%2Cswell_wave_peak_period%2Csecondary_swell_wave_height%2Csecondary_swell_wave_direction%2Csecondary_swell_wave_period%2Ctertiary_swell_wave_height%2Ctertiary_swell_wave_direction%2Ctertiary_swell_wave_period%2Csea_level_height_msl%2Csea_surface_temperature%2Cocean_current_velocity%2Cocean_current_direction&past_days=7&forecast_days=2&timezone=UTC&cell_selection=sea'}


,valid_at_utc,fetched_at_utc,wave_height,wave_period,wave_direction,forecast_status
180,2026-11-08T12:00:00Z,2026-11-07T07:06:34.428469+00:00,2.46,9.90,7,future
181,2026-11-08T13:00:00Z,2026-11-07T07:06:34.428469+00:00,2.48,9.90,7,future
182,2026-11-08T14:00:00Z,2026-11-07T07:06:34.428469+00:00,2.48,9.85,8,future
183,2026-11-08T15:00:00Z,2026-11-07T07:06:34.428469+00:00,2.50,9.80,8,future
184,2026-11-08T16:00:00Z,2026-11-07T07:06:34.428469+00:00,2.52,9.75,9,future
185,2026-11-08T17:00:00Z,2026-11-07T07:06:34.428469+00:00,2.52,9.65,10,future
186,2026-11-08T18:00:00Z,2026-11-07T07:06:34.428469+00:00,2.54,9.60,11,future
187,2026-11-08T19:00:00Z,2026-11-07T07:06:34.428469+00:00,2.54,9.55,11,future
188,2026-11-08T20:00:00Z,2026-11-07T07:06:34.428469+00:00,2.54,9.50,12,future
189,2026-11-08T21:00:00Z,2026-11-07T07:06:34.428469+00:00,2.54,9.45,12,future


## 7. PacIOOS ERDDAP — kiểm thử trước, cho phép backfill khi truy cập thành công

Dataset `cdip_wave_agg` có các biến `station_id, time, waveHs, waveTp, waveTa, waveDp, waveFlagPrimary`; lọc `station_id="106"`. Đây là kênh cung cấp dữ liệu quan trắc **cùng một phao** với NOAA 51201. Thời gian `time` được mô tả là thời điểm bắt đầu mẫu 30 phút; không nên trộn timestamp 1:1 với NOAA mà không kiểm tra quy ước.

Dịch vụ PacIOOS/CDIP đôi khi có lỗi 403, 404, 429 hoặc quá tải; mặc định chỉ probe 7 ngày. Không tự động lấy 2005–nay từ ERDDAP cho đến khi xác nhận độ bao phủ và tốc độ truy xuất thực tế. `PACIOOS_MODE='full'` sẽ thực hiện backfill theo tháng, với cơ chế dừng sau lỗi liên tiếp.

In [9]:
def pacioos_url(begin: date, finish: date) -> str:
    # Constraints ERDDAP: station_id=..., time>=... and time< next_day.
    # quote từng phần để không nhầm dấu so sánh vào cú pháp của requests.
    start = f"{begin.isoformat()}T00:00:00Z"
    end_exclusive = f"{(finish + timedelta(days=1)).isoformat()}T00:00:00Z"
    fields = ",".join(PAC_VARS)
    rules = [f'station_id="106"', f"time>={start}", f"time<{end_exclusive}"]
    encoded = "&".join(quote(rule, safe="=:") for rule in rules)
    return PACIOOS_URL + "?" + fields + "&" + encoded


def parse_pacioos_csv(raw: str) -> pd.DataFrame:
    if not raw.strip() or raw.lstrip().startswith("<html"):
        raise ValueError("ERDDAP không trả CSV hợp lệ")
    # ERDDAP CSV: header + hàng units + dữ liệu
    frame = pd.read_csv(StringIO(raw), skiprows=[1], dtype={"station_id": str})
    if "time" not in frame.columns or "station_id" not in frame.columns:
        raise ValueError(f"ERDDAP CSV thiếu time/station_id; cols={frame.columns.tolist()}")
    if not set(PAC_VARS).issubset(frame.columns):
        raise ValueError(f"ERDDAP trả thiếu biến mong đợi: {set(PAC_VARS)-set(frame.columns)}")
    frame = frame.loc[frame["station_id"].astype(str).str.lstrip("0") == "106"].copy()
    frame["timestamp_utc"] = pd.to_datetime(frame.pop("time"), utc=True, errors="coerce")
    frame = frame.dropna(subset=["timestamp_utc"])
    frame["timestamp_utc"] = frame["timestamp_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    for field in ["waveHs", "waveTp", "waveTa", "waveDp", "waveFlagPrimary"]:
        frame[field] = pd.to_numeric(frame[field], errors="coerce")
        frame[field] = frame[field].mask(frame[field] <= -900)  # sentinel -999.99
    frame["source"] = "pacioos_cdip_106"
    frame["data_type"] = "buoy_observation"
    return frame.drop_duplicates(subset=["timestamp_utc"]).sort_values("timestamp_utc")


def fetch_pacioos_period(begin: date, finish: date) -> dict:
    year, month = begin.year, begin.month
    key = f"{begin.isoformat()}_{finish.isoformat()}"
    try:
        text, url = get_url(pacioos_url(begin, finish))
        frame = parse_pacioos_csv(text)
        raw = raw_once(PAC_ROOT / "data-raw" / str(year), key, text, ".csv")
        added = 0
        for (yr, mo), chunk in frame.groupby([
            pd.to_datetime(frame["timestamp_utc"], utc=True).dt.year,
            pd.to_datetime(frame["timestamp_utc"], utc=True).dt.month,
        ]):
            plus, _ = append_csv_unique(chunk, month_path(PAC_ROOT, int(yr), int(mo)))
            added += plus
        result = {"period": key, "status": "ok", "rows": len(frame), "new_rows": added,
                  "raw": str(raw.relative_to(ROOT)), "url": url}
        log_run(PAC_ROOT, result)
        return result
    except (requests.RequestException, RuntimeError, ValueError, KeyError, OSError, pd.errors.ParserError) as exc:
        result = {"period": key, "status": "error", "new_rows": 0,
                  "error": f"{type(exc).__name__}: {str(exc)[:320]}"}
        log_run(PAC_ROOT, result)
        return result


def run_pacioos():
    if PACIOOS_MODE == "off":
        print("Đã tắt PacIOOS")
        return pd.DataFrame()
    if PACIOOS_MODE == "probe":
        today = utc_now().date()
        results = [fetch_pacioos_period(today - timedelta(days=7), today)]
    elif PACIOOS_MODE == "full":
        results = []
        errors = 0
        for year, month, begin, end, full_end in month_chunks(HIST_START, utc_now().date()):
            # Chỉ cache tháng đầy đủ, còn tháng đang mở cần thử lại trong các lượt chạy sau.
            manifest = safe_json(PAC_ROOT / "manifest.json", {"months": {}})
            k = f"{year:04d}-{month:02d}"
            if manifest.get("months", {}).get(k, {}).get("complete") and not RECHECK_FINALIZED_MONTHS:
                results.append({"period": k, "status": "cached", "new_rows": 0})
                continue
            result = fetch_pacioos_period(begin, end)
            results.append(result)
            if result["status"] == "error":
                errors += 1
                print("[PACIOOS ERROR]", result.get("error"))
                if errors >= MAX_CONSECUTIVE_SOURCE_ERRORS:
                    print("[STOP] Nhiều lỗi liên tiếp, dừng PacIOOS full backfill")
                    break
            else:
                errors = 0
                manifest.setdefault("months", {})[k] = {"complete": end == full_end,
                   "last_success_utc": utc_now().isoformat(), "rows": result.get("rows", 0)}
                save_manifest(PAC_ROOT, manifest)
                if REQUEST_PAUSE_SECONDS:
                    time.sleep(REQUEST_PAUSE_SECONDS)
    else:
        raise ValueError("PACIOOS_MODE chỉ chấp nhận off/probe/full")
    return pd.DataFrame(results)

In [10]:
pacioos_results = run_pacioos()
if not pacioos_results.empty:
    display(pacioos_results.tail(12))

,period,status,new_rows,error
0,2026-10-31_2026-11-07,error,0,RuntimeError: Lấy dữ liệu thất bại: ConnectTim...


## 8. Tổng hợp số bản ghi đã tích lũy và kiểm tra không có duplicate trong từng dataset

In [11]:
def summarize_csv(root: Path, key: str) -> pd.DataFrame:
    rows = []
    for csv_path in sorted((root / "data-csv").glob("*/*.csv")):
        d = pd.read_csv(csv_path, usecols=[key], dtype=str)
        unique_count = d[key].nunique(dropna=False)
        if len(d) != unique_count:
            raise AssertionError(f"Trùng {key} trong {csv_path}")
        rows.append({"file": str(csv_path.relative_to(ROOT)), "records": len(d),
                     "unique": unique_count, "first": d[key].min(), "last": d[key].max()})
    return pd.DataFrame(rows)

for root, key in [(HIST_ROOT, "timestamp_utc"), (MAR_ROOT, "revision_key"), (PAC_ROOT, "timestamp_utc")]:
    print("\n===", root.name, "===")
    s = summarize_csv(root, key)
    if s.empty:
        print("Chưa có CSV, hãy kiểm tra log và endpoint")
    else:
        print("CSV:", len(s), "| tổng bản ghi:", int(s.records.sum()))
        display(s.tail(8))

print("\nThư mục dữ liệu cuối cùng:", ROOT.resolve())
print("Log của từng nguồn nằm ở: <source>/run-logs/run_log.jsonl")


=== open_meteo_historical ===
CSV: 262 | tổng bản ghi: 191376


,file,records,unique,first,last
254,open_meteo_historical/data-csv/2026/February-d...,672,672,2026-02-01T00:00:00Z,2026-02-28T23:00:00Z
255,open_meteo_historical/data-csv/2026/January-da...,744,744,2026-01-01T00:00:00Z,2026-01-31T23:00:00Z
256,open_meteo_historical/data-csv/2026/July-data-...,744,744,2026-07-01T00:00:00Z,2026-07-31T23:00:00Z
257,open_meteo_historical/data-csv/2026/June-data-...,720,720,2026-06-01T00:00:00Z,2026-06-30T23:00:00Z
258,open_meteo_historical/data-csv/2026/March-data...,744,744,2026-03-01T00:00:00Z,2026-03-31T23:00:00Z
259,open_meteo_historical/data-csv/2026/May-data-2...,744,744,2026-05-01T00:00:00Z,2026-05-31T23:00:00Z
260,open_meteo_historical/data-csv/2026/October-da...,744,744,2026-10-01T00:00:00Z,2026-10-31T23:00:00Z
261,open_meteo_historical/data-csv/2026/September-...,720,720,2026-09-01T00:00:00Z,2026-09-30T23:00:00Z



=== open_meteo_marine ===
CSV: 2 | tổng bản ghi: 216


,file,records,unique,first,last
0,open_meteo_marine/data-csv/2026/November-versi...,192,192,2026-11-01T00:00:00Z|49bf113a02657c33e39b1a40e...,2026-11-08T23:00:00Z|277623650d246dee52c145240...
1,open_meteo_marine/data-csv/2026/October-versio...,24,24,2026-10-31T00:00:00Z|9e22f331e44c9b827d9df1156...,2026-10-31T23:00:00Z|7f425f0883afd585eefca7811...



=== pacioos_erddap ===
Chưa có CSV, hãy kiểm tra log và endpoint

Thư mục dữ liệu cuối cùng: /run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data/other_apis
Log của từng nguồn nằm ở: <source>/run-logs/run_log.jsonl


## 9. Xem cây thư mục dữ liệu đã tạo

In [12]:
def preview_tree(folder: Path, levels=4, limit_per_folder=8):
    def walk(p: Path, level=0):
        if level > levels:
            return
        children = sorted(p.iterdir(), key=lambda v: (v.is_file(), v.name))
        for item in children[:limit_per_folder]:
            print("    " * level + ("└─ " if item == children[-1] else "├─ ") + item.name + ("/" if item.is_dir() else ""))
            if item.is_dir():
                walk(item, level + 1)
        if len(children) > limit_per_folder:
            print("    " * level + f"... và {len(children) - limit_per_folder} mục khác")
print(ROOT)
preview_tree(ROOT, levels=3)

/run/media/sylein/DRIVER F/23115053122114_K23_PhanVanKhanh/College_Year04/126/Kho-du-lieu/repo/waimea_data/other_apis


## 10. Kết luận và bước chuyển sang Backend

- NOAA vẫn là **nguồn quan trắc chính** và có notebook riêng của bạn. Trong hệ thống, lưu `source` để phân biệt nguồn thật và model.
- Open-Meteo ERA5 lịch sử từ năm 2005 được lưu theo tháng và **không ghi đè** khi chạy lại. Do tái phân tích có độ trễ, ngày gần nhất có thể chưa có dữ liệu; không được xem lỗi trễ này như sự cố Real-Time.
- Open-Meteo Marine lưu tất cả 22 biến hourly nếu API trả về, với bản ghi mới theo **revision hash**. Chạy lại sau 30 phút hoặc theo scheduler để phát hiện thay đổi.
- PacIOOS/CDIP 106 là kênh thay thế quan trắc, **không phải một phao khác**. Chạy `probe` trước khi đổi sang full.
- Chưa có cơ sở kết luận tất cả API cung cấp toàn bộ dữ liệu 2005–2026; notebook ghi log khi nguồn không có dữ liệu.
- File raw và CSV là dữ liệu **Bronze cục bộ**, chưa tự upload lên Google Cloud Storage hoặc ghi Cloud SQL. Khi triển khai, service Data Ingestion sẽ đọc các file/cursor này, upload raw lên GCS và gửi bản ghi chuẩn hóa sang Database API.
- **Không chạy đồng thời hai notebook ghi vào cùng thư mục `other_apis/`**. Nếu cần chạy đa tiến trình, chuyển sang DB với unique constraint/transaction hoặc thêm file lock.

### Để cập nhật mới nhất

Bạn chỉ cần mở notebook này rồi **Run All**. Các tháng đã hoàn chỉnh sẽ được bỏ qua, chỉ tháng còn mở và bản Marine mới được gọi thêm. Với chế độ theo dõi 24/7, chạy bộ hàm trên bằng scheduler trong Backend, không phụ thuộc Jupyter Notebook.